# synthkit worked example: UCI Adult Census Income

This is the argument for the whole package, run against real data. Faker-style sampling
(every column drawn independently) produces rows that look individually plausible and are
collectively meaningless: `age` and `education-num` stop correlating, `hours-per-week` stops
predicting `income`. synthkit's Gaussian copula is built specifically to not throw that
structure away.

32,561 rows, no nulls, a mix of continuous and categorical columns -- a clean first case.

In [1]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

import synthkit as sk

DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "adult.data"
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income",
]

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    subprocess.run(["curl", "-sL", "-o", str(DATA_PATH), DATA_URL], check=True)

real = pd.read_csv(DATA_PATH, names=COLUMNS, skipinitialspace=True)
real = real[real["age"] != "?"].dropna()
real["income_over_50k"] = (real["income"] == ">50K").astype(int)
real = real[["age", "education-num", "hours-per-week", "income", "income_over_50k"]]
real.head()

,age,education-num,hours-per-week,income,income_over_50k
0,39,13,40,<=50K,0
1,50,13,13,<=50K,0
2,38,9,40,<=50K,0
3,53,7,40,<=50K,0
4,28,13,40,<=50K,0


## Fit a profile and emit synthetic rows

In [2]:
profile = sk.fit(real)
synthetic = sk.emit(profile, n=10_000, seed=0)
synthetic.head()

,age,education-num,hours-per-week,income,income_over_50k
0,39.0,9,45.0,<=50K,0
1,43.0,13,50.0,<=50K,0
2,28.0,10,7.0,<=50K,0
3,27.0,9,40.0,<=50K,0
4,35.0,13,40.0,<=50K,0


## The comparison

Faker-style data here means every column shuffled independently -- exactly what sampling
each column's marginal with no joint model produces. Compare its correlations against real
data and against synthkit's output.

In [3]:
def correlations(df):
    age_edu = np.corrcoef(df["age"].astype(float), df["education-num"].astype(float))[0, 1]
    hours_income = np.corrcoef(
        df["hours-per-week"].astype(float), pd.to_numeric(df["income_over_50k"])
    )[0, 1]
    return age_edu, hours_income


rng = np.random.default_rng(0)
faker = pd.DataFrame({col: rng.permutation(real[col].to_numpy()) for col in real.columns})

real_age_edu, real_hours_income = correlations(real)
faker_age_edu, faker_hours_income = correlations(faker)
synth_age_edu, synth_hours_income = correlations(synthetic)

pd.DataFrame(
    {
        "corr(age, education-num)": [real_age_edu, faker_age_edu, synth_age_edu],
        "corr(hours, income)": [real_hours_income, faker_hours_income, synth_hours_income],
    },
    index=["real", "Faker-style", "synthkit"],
).round(3)

,"corr(age, education-num)","corr(hours, income)"
real,0.037,0.230
Faker-style,0.001,0.000
synthkit,0.065,0.183


## The punchline

A test whose correctness depends on the correlation between `hours-per-week` and `income`
(mean hours worked, high income vs low income group) should pass on real data, pass on
synthkit's output, and fail on Faker-style data -- because that correlation is exactly what
independent-column sampling throws away.

In [4]:
def passes_joint_distribution_test(df):
    income = pd.to_numeric(df["income_over_50k"])
    hours = df["hours-per-week"].astype(float)
    high = hours[income == 1].mean()
    low = hours[income == 0].mean()
    return (high - low) > 2.0


for name, dataset in [("real", real), ("synthkit", synthetic), ("Faker-style", faker)]:
    result = "PASS" if passes_joint_distribution_test(dataset) else "FAIL"
    print(f"{name:12s} {result}")

real         PASS
synthkit     PASS
Faker-style  FAIL


## Privacy check

`dcr_ratio >= 1.0` means synthetic rows sit at least as far from real training rows as a
real holdout naturally does -- the profile isn't just memorizing rows it was fit on.

In [5]:
report = sk.check(synthetic, profile, real=real, min_dcr_ratio=0.5)
print(f"dcr_ratio: {report.dcr_ratio:.3f}")
print(f"exact_matches: {report.exact_matches}")
print(f"passed: {report.passed}")

dcr_ratio: 1.000
exact_matches: 0
passed: True
